In [ ]:
import json
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

BASE_DIR = Path.cwd().parent.parent.parent

OUTPUT_DIR1 = BASE_DIR / "outputs"
SPLIT_DIR = OUTPUT_DIR1 / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
OUTPUT_DIR = BASE_DIR / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model_state_dict.pt"
BEST_VAL_LOGITS_PATH = OUTPUT_DIR / "best_val_logits.npy"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history.csv"

TEXT_PATH = OUTPUT_DIR1 / "y_logits_camembert_run2_epochs3.npy"

STACK_MODEL_PATH = OUTPUT_DIR / "final_stacking_model.pkl"
STACK_FEATURES_PATH = OUTPUT_DIR / "stacking_features.npy"
STACK_PRED_PATH = OUTPUT_DIR / "stacking_predictions.npy"
STACK_REPORT_PATH = OUTPUT_DIR / "stacking_report.csv"
STACK_METADATA_PATH = OUTPUT_DIR / "stacking_metadata.json"
STACK_PROBA_PATH = OUTPUT_DIR / "stacking_probabilities.npy"

multi_logits = np.load(BEST_VAL_LOGITS_PATH)
text_logits = np.load(TEXT_PATH)

print("multimodal logits shape:", multi_logits.shape)
print("text logits shape      :", text_logits.shape)

if multi_logits.shape != text_logits.shape:
    raise ValueError(
        f"Shape mismatch: multimodal={multi_logits.shape}, text={text_logits.shape}"
    )

val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

if len(y_true) != multi_logits.shape[0]:
    raise ValueError(
        f"Validation length mismatch: y_true={len(y_true)}, logits={multi_logits.shape[0]}"
    )

# ============================================================
# Stacking input features
# ============================================================
# Burada iki modelin logitslerini yan yana koyuyoruz.
# 27 class + 27 class = 54 feature
X = np.concatenate([multi_logits, text_logits], axis=1)

print("stacking feature shape :", X.shape)

# ============================================================
# Final stacking model
# ============================================================
stack_model = LogisticRegression(
    max_iter=3000,
    solver="lbfgs"
)

stack_model.fit(X, y_true)

pred = stack_model.predict(X)
proba = stack_model.predict_proba(X)

macro_f1 = f1_score(y_true, pred, average="macro")
acc = accuracy_score(y_true, pred)

print("\nFINAL STACKING RESULTS")
print("Accuracy :", round(acc, 6))
print("Macro F1 :", round(macro_f1, 6))

# ============================================================
# Save artifacts
# ============================================================
joblib.dump(stack_model, STACK_MODEL_PATH)
np.save(STACK_FEATURES_PATH, X)
np.save(STACK_PRED_PATH, pred)
np.save(STACK_PROBA_PATH, proba)

report_df = pd.DataFrame(
    classification_report(y_true, pred, output_dict=True, zero_division=0)
).transpose()
report_df.to_csv(STACK_REPORT_PATH)

with open(STACK_METADATA_PATH, "w") as f:
    json.dump(
        {
            "model_type": "LogisticRegression_stacking",
            "macro_f1": float(macro_f1),
            "accuracy": float(acc),
            "multimodal_shape": list(multi_logits.shape),
            "text_shape": list(text_logits.shape),
            "stacking_feature_shape": list(X.shape),
        },
        f,
        indent=2
    )

print("\nSaved:")
print(STACK_MODEL_PATH)
print(STACK_FEATURES_PATH)
print(STACK_PRED_PATH)
print(STACK_PROBA_PATH)
print(STACK_REPORT_PATH)
print(STACK_METADATA_PATH)

In [ ]:
import json
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

BASE_DIR = Path.cwd().parent.parent.parent

OUTPUT_DIR1 = BASE_DIR / "outputs"
SPLIT_DIR = OUTPUT_DIR1 / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
OUTPUT_DIR = BASE_DIR / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_VAL_LOGITS_PATH = OUTPUT_DIR / "best_val_logits.npy"
TEXT_PATH = OUTPUT_DIR1 / "y_logits_camembert_run2_epochs3.npy"

STACK_MODEL_PATH = OUTPUT_DIR / "final_stacking_model_split.pkl"
STACK_FEATURES_TRAIN_PATH = OUTPUT_DIR / "stacking_X_meta_train.npy"
STACK_FEATURES_VAL_PATH = OUTPUT_DIR / "stacking_X_meta_val.npy"
STACK_Y_TRAIN_PATH = OUTPUT_DIR / "stacking_y_meta_train.npy"
STACK_Y_VAL_PATH = OUTPUT_DIR / "stacking_y_meta_val.npy"
STACK_PRED_PATH = OUTPUT_DIR / "stacking_predictions_meta_val.npy"
STACK_PROBA_PATH = OUTPUT_DIR / "stacking_probabilities_meta_val.npy"
STACK_REPORT_PATH = OUTPUT_DIR / "stacking_report_meta_val.csv"
STACK_METADATA_PATH = OUTPUT_DIR / "stacking_metadata_meta_val.json"

multi_logits = np.load(BEST_VAL_LOGITS_PATH)
text_logits = np.load(TEXT_PATH)

print("multimodal logits shape:", multi_logits.shape)
print("text logits shape      :", text_logits.shape)

if multi_logits.shape != text_logits.shape:
    raise ValueError(
        f"Shape mismatch: multimodal={multi_logits.shape}, text={text_logits.shape}"
    )

val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r", encoding="utf-8") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

if len(y_true) != multi_logits.shape[0]:
    raise ValueError(
        f"Validation length mismatch: y_true={len(y_true)}, logits={multi_logits.shape[0]}"
    )

X = np.concatenate([multi_logits, text_logits], axis=1)

print("stacking feature shape:", X.shape)

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X,
    y_true,
    test_size=0.3,
    random_state=42,
    stratify=y_true
)

print("X_meta_train:", X_meta_train.shape)
print("X_meta_val  :", X_meta_val.shape)

stack_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        C=1.0
    ))
])

stack_model.fit(X_meta_train, y_meta_train)

pred = stack_model.predict(X_meta_val)
proba = stack_model.predict_proba(X_meta_val)

macro_f1 = f1_score(y_meta_val, pred, average="macro")
weighted_f1 = f1_score(y_meta_val, pred, average="weighted")
acc = accuracy_score(y_meta_val, pred)

print("\nSTACKING RESULTS ON META-VAL")
print("Accuracy    :", round(acc, 6))
print("Macro F1    :", round(macro_f1, 6))
print("Weighted F1 :", round(weighted_f1, 6))

report_df = pd.DataFrame(
    classification_report(y_meta_val, pred, output_dict=True, zero_division=0)
).transpose()

joblib.dump(stack_model, STACK_MODEL_PATH)
np.save(STACK_FEATURES_TRAIN_PATH, X_meta_train)
np.save(STACK_FEATURES_VAL_PATH, X_meta_val)
np.save(STACK_Y_TRAIN_PATH, y_meta_train)
np.save(STACK_Y_VAL_PATH, y_meta_val)
np.save(STACK_PRED_PATH, pred)
np.save(STACK_PROBA_PATH, proba)
report_df.to_csv(STACK_REPORT_PATH)

with open(STACK_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_type": "LogisticRegression_stacking_meta_split",
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "multimodal_shape": list(multi_logits.shape),
            "text_shape": list(text_logits.shape),
            "stacking_feature_shape": list(X.shape),
            "X_meta_train_shape": list(X_meta_train.shape),
            "X_meta_val_shape": list(X_meta_val.shape),
            "test_size": 0.3,
            "random_state": 42
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print(STACK_MODEL_PATH)
print(STACK_FEATURES_TRAIN_PATH)
print(STACK_FEATURES_VAL_PATH)
print(STACK_Y_TRAIN_PATH)
print(STACK_Y_VAL_PATH)
print(STACK_PRED_PATH)
print(STACK_PROBA_PATH)
print(STACK_REPORT_PATH)
print(STACK_METADATA_PATH)